<div dir="rtl">

# الدرس 40: محاربة الإفراط في الشبكات العصبية

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

## تجربة: أرقام بخط اليد، وأجوبة فيها أخطاء

</div>

In [ ]:
import copy
import numpy as np
import torch
from torch import nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

X, y = load_digits(return_X_y=True)          # 1797 images, 8x8 pixels, digits 0-9
X = X / 16.0                                  # pixels 0..16 -> 0..1
X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, train_size=300, stratify=y, random_state=0)
X_val, X_te, y_val, y_te = train_test_split(X_rest, y_rest, test_size=0.5,
                                            stratify=y_rest, random_state=0)

def to_t(a, dtype=torch.float32):
    return torch.tensor(a, dtype=dtype)

X_tr, X_val, X_te = to_t(X_tr), to_t(X_val), to_t(X_te)
y_tr, y_val, y_te = to_t(y_tr, torch.long), to_t(y_val, torch.long), to_t(y_te, torch.long)

# replace 20% of the training labels with random digits
g = torch.Generator().manual_seed(2)
y_noisy = y_tr.clone()
flip = torch.rand(len(y_noisy), generator=g) < 0.2
y_noisy[flip] = torch.randint(0, 10, (int(flip.sum()),), generator=g)
print("train / val / test:", len(X_tr), len(X_val), len(X_te))
print("wrong training labels:", int((y_noisy != y_tr).sum()))

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 8, figsize=(9, 2.8))
for ax, i in zip(axes.ravel(), range(16)):
    ax.imshow(X_tr[i].reshape(8, 8), cmap="gray_r")
    wrong = bool(y_noisy[i] != y_tr[i])
    style = {"color": "C3", "fontweight": "bold"} if wrong else {}
    ax.set_title(f"label {int(y_noisy[i])}" + (" ✗" if wrong else ""), fontsize=9, **style)
    ax.axis("off")
plt.tight_layout()
plt.show()

<div dir="rtl">

### دالة التدريب، مع الإيقاف المبكر

</div>

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def make_model(p_drop=0.0):
    torch.manual_seed(0)
    return nn.Sequential(
        nn.Linear(64, 256), nn.ReLU(), nn.Dropout(p_drop),
        nn.Linear(256, 256), nn.ReLU(), nn.Dropout(p_drop),
        nn.Linear(256, 10))

def evaluate(model, X, y):
    model.eval()                                          # dropout OFF
    with torch.no_grad():
        logits = model(X)
        return loss_fn(logits, y).item(), (logits.argmax(1) == y).float().mean().item()

def train(model, opt, labels, epochs=200, patience=None, batch_size=32):
    g = torch.Generator().manual_seed(0)
    history, best = [], {"loss": float("inf")}
    for epoch in range(1, epochs + 1):
        model.train()                                     # dropout ON
        for idx in torch.randperm(len(X_tr), generator=g).split(batch_size):
            loss = loss_fn(model(X_tr[idx]), labels[idx])
            opt.zero_grad(); loss.backward(); opt.step()
        val_loss, val_acc = evaluate(model, X_val, y_val)
        history.append((evaluate(model, X_tr, labels)[1], val_loss, val_acc))
        if val_loss < best["loss"]:                        # remember the best weights
            best = {"loss": val_loss, "epoch": epoch, "acc": val_acc,
                    "state": copy.deepcopy(model.state_dict())}
        elif patience and epoch - best["epoch"] >= patience:
            break                                         # early stopping
    return np.array(history), best

<div dir="rtl">

### الأجوبة العشوائية: الشبكة تحفظ الضجيج

</div>

In [ ]:
y_random = torch.randint(0, 10, (len(X_tr),), generator=torch.Generator().manual_seed(1))
model = make_model()
history, _ = train(model, torch.optim.Adam(model.parameters(), lr=1e-3), y_random, epochs=300)
first = int(np.argmax(history[:, 0] >= 0.999)) + 1
print(f"train acc {history[-1, 0]:.3f} (100% from epoch {first}), val acc {history[-1, 2]:.3f}")

<div dir="rtl">

### ثلاث طرق على الأجوبة المشوّشة

</div>

In [ ]:
configs = {
    "baseline":         dict(p_drop=0.0, weight_decay=0.0),
    "weight decay 1.0": dict(p_drop=0.0, weight_decay=1.0),
    "dropout 0.5":      dict(p_drop=0.5, weight_decay=0.0),
}
results = {}
for name, cfg in configs.items():
    model = make_model(cfg["p_drop"])
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=cfg["weight_decay"])
    history, best = train(model, opt, y_noisy)
    results[name] = (history, best)
    print(f"{name:16}: train acc {history[-1, 0]:.3f} | final val acc {history[-1, 2]:.3f} | "
          f"best epoch {best['epoch']:3}, val acc there {best['acc']:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for name, (h, best) in results.items():
    line, = axes[0].plot(range(1, len(h) + 1), h[:, 1], label=name)
    axes[0].plot(best["epoch"], best["loss"], "o", color=line.get_color())
    axes[1].plot(range(1, len(h) + 1), h[:, 2], label=name)
axes[0].set_ylabel("validation loss"); axes[1].set_ylabel("validation accuracy")
for ax in axes:
    ax.set_xlabel("epoch"); ax.grid(True, alpha=0.4)
axes[0].legend()
plt.tight_layout()
plt.show()

<div dir="rtl">

### الإسقاط مع الإيقاف المبكر، ثم الاختبار مرة واحدة

</div>

In [ ]:
model = make_model(0.5)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.0)
history, best = train(model, opt, y_noisy, patience=20)
print(f"stopped after epoch {len(history)}, best epoch {best['epoch']}")
model.load_state_dict(best["state"])
test_loss, test_acc = evaluate(model, X_te, y_te)
print(f"test accuracy: {test_acc:.3f}")

<div dir="rtl">

### الإسقاط في الوضعين

</div>

In [ ]:
drop = nn.Dropout(p=0.25)
h = torch.tensor([2.0, 4.0, 6.0, 8.0])
torch.manual_seed(9)
print("train mode:", drop(h))
drop.eval()
print("eval mode :", drop(h))

<div dir="rtl">

## تمرين: ما أفضل احتمال للإسقاط؟

جرّب احتمالات الإسقاط 0.2، و 0.5، و 0.8، و 0.95 على نفس البيانات المشوّشة، بدون تقليص أوزان.

1. لكل احتمال: اطبع دقة التدريب النهائية، ودقة التحقق النهائية، ودقة التحقق في أفضل حقبة.
2. أي احتمال أفضل هنا؟ وماذا حدث مع أكبر احتمال؟

</div>

In [ ]:
# اكتب حلّك هنا
